In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/khusheeranjan/mscadaa/MSCAD.csv


In [2]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/khusheeranjan/mscadaa/MSCAD.csv")  # or read_excel, etc.

print(df.shape)
print(df.dtypes)
print(df.head())
print(df.describe(include='all'))
print(df.isnull().sum())

(128799, 67)
'Flow Duration'       int64
'Tot Fwd Pkts'        int64
'Tot Bwd Pkts'        int64
'TotLen Fwd Pkts'     int64
'TotLen Bwd Pkts'     int64
                      ...  
'Idle Mean'           int64
'Idle Std'            int64
'Idle Max'            int64
'Idle Min'            int64
Label                object
Length: 67, dtype: object
   'Flow Duration'  'Tot Fwd Pkts'  'Tot Bwd Pkts'  'TotLen Fwd Pkts'  \
0             1518               2               5                110   
1             5894               4               8                168   
2              272               1               1                  0   
3             2611               4               8                322   
4              294               1               1                  0   

   'TotLen Bwd Pkts'  'Fwd Pkt Len Max'  'Fwd Pkt Len Min'  \
0                377                110                  0   
1               4498                168                  0   
2                  0        

In [3]:
# Clean up column names (remove stray quotes/whitespace)
df.columns = df.columns.str.replace("'", "").str.strip()

print(df.columns.tolist())

# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check for infinite values (common in this type of dataset)
import numpy as np
print("Inf values:", np.isinf(df.select_dtypes(include=[np.number])).sum().sum())

['Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Init Bwd Win Byts', 'Fwd Act Data Pkts', 'Active Mean

In [4]:
# Drop duplicates
df = df.drop_duplicates()
print("New shape:", df.shape)

# Check class balance
print(df['Label'].value_counts())

# Check for constant columns (no variance - useless for ML)
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Constant columns:", constant_cols)

New shape: (92035, 67)
Label
Brute_Force    62020
Normal         26209
Port_Scan       3292
HTTP_DDoS        441
ICMP_Flood        45
Web_Crwling       28
Name: count, dtype: int64
Constant columns: []


In [5]:
# Check for negative values (shouldn't exist in flow data)
numeric_cols = df.select_dtypes(include=[np.number]).columns
neg_counts = (df[numeric_cols] < 0).sum()
print("Columns with negative values:\n", neg_counts[neg_counts > 0])

# Check extreme skew/outliers in key rate columns
print(df[['Flow Byts/s', 'Flow Pkts/s']].describe())

# Check how many rows have 0 duration but nonzero packets (anomalous flows)
zero_dur = df[(df['Flow Duration'] == 0)]
print("Rows with 0 flow duration:", len(zero_dur))

Columns with negative values:
 Flow IAT Min            19
Init Bwd Win Byts    18160
dtype: int64
        Flow Byts/s    Flow Pkts/s
count  9.203500e+04   92035.000000
mean   2.479902e+05    1028.237520
std    6.283041e+05    3914.039578
min    0.000000e+00       0.000000
25%    0.000000e+00      19.872650
50%    3.325566e+04     521.421700
75%    4.812886e+05    1283.628350
max    4.561039e+07  500000.000000
Rows with 0 flow duration: 82


In [6]:
# Drop rows with negative Flow IAT Min (genuine data error)
df = df[df['Flow IAT Min'] >= 0]
print("Shape after removing bad IAT rows:", df.shape)

# Confirm Init Bwd Win Byts negative values are all -1 (sentinel) not random negatives
print(df['Init Bwd Win Byts'][df['Init Bwd Win Byts'] < 0].value_counts())

# Check label distribution of zero-duration flows (are they meaningful or junk?)
print(df[df['Flow Duration'] == 0]['Label'].value_counts())

Shape after removing bad IAT rows: (92016, 67)
Init Bwd Win Byts
-1    18160
Name: count, dtype: int64
Label
Normal    82
Name: count, dtype: int64


In [7]:
# Separate features and target
X = df.drop('Label', axis=1)
y = df['Label']

# Encode labels (for ML models)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(dict(zip(le.classes_, range(len(le.classes_)))))

# Check highly correlated features (redundant info)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [(col, row, upper.loc[row, col]) for col in upper.columns for row in upper.index if upper.loc[row, col] > 0.95]
print("Number of highly correlated pairs (>0.95):", len(high_corr))
print(high_corr[:10])

{'Brute_Force': 0, 'HTTP_DDoS': 1, 'ICMP_Flood': 2, 'Normal': 3, 'Port_Scan': 4, 'Web_Crwling': 5}
Number of highly correlated pairs (>0.95): 42
[('TotLen Fwd Pkts', 'Tot Fwd Pkts', np.float64(0.9970266463137211)), ('Bwd Pkt Len Min', 'Fwd Pkt Len Min', np.float64(0.9701828311999552)), ('Bwd Pkt Len Std', 'Bwd Pkt Len Max', np.float64(0.9845165069573694)), ('Flow IAT Min', 'Flow IAT Mean', np.float64(0.958380781015207)), ('Fwd IAT Min', 'Fwd IAT Mean', np.float64(0.9544138172301531)), ('Fwd Header Len', 'Tot Fwd Pkts', np.float64(0.9978575148365756)), ('Fwd Header Len', 'TotLen Fwd Pkts', np.float64(0.9991374166929541)), ('Bwd Header Len', 'Tot Bwd Pkts', np.float64(0.9775104339584307)), ('Bwd Pkts/s', 'Flow Pkts/s', np.float64(0.9524475174274998)), ('Pkt Len Min', 'Fwd Pkt Len Min', np.float64(0.9916141199579218))]


In [8]:
# Build a set of columns to drop based on correlation threshold
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = set()
for col in upper.columns:
    for row in upper.index:
        if upper.loc[row, col] > 0.95:
            # drop the second one in the pair (col), keep row
            to_drop.add(col)

print("Columns to drop:", to_drop)
print("Number of columns to drop:", len(to_drop))

X_reduced = X.drop(columns=to_drop)
print("New feature shape:", X_reduced.shape)

Columns to drop: {'Bwd Header Len', 'Bwd Pkts/s', 'PSH Flag Cnt', 'Fwd Act Data Pkts', 'Bwd Seg Size Avg', 'Idle Max', 'Bwd Pkt Len Std', 'Idle Min', 'URG Flag Cnt', 'ECE Flag Cnt', 'Subflow Bwd Byts', 'Pkt Len Std', 'Pkt Len Mean', 'Subflow Fwd Byts', 'Flow IAT Min', 'Bwd Pkt Len Min', 'Subflow Fwd Pkts', 'Pkt Len Min', 'Subflow Bwd Pkts', 'Fwd IAT Min', 'TotLen Fwd Pkts', 'Fwd Header Len', 'Fwd Seg Size Avg', 'Pkt Size Avg'}
Number of columns to drop: 24
New feature shape: (92016, 42)


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Stratified split to preserve class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# Check class distribution preserved
import collections
print("Train class dist:", collections.Counter(y_train))
print("Test class dist:", collections.Counter(y_test))

# Scale features (fit on train only, to avoid data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled train sample:\n", X_train_scaled[:2])

Train shape: (73612, 42) Test shape: (18404, 42)
Train class dist: Counter({np.int64(0): 49615, np.int64(3): 20952, np.int64(4): 2634, np.int64(1): 353, np.int64(2): 36, np.int64(5): 22})
Test class dist: Counter({np.int64(0): 12405, np.int64(3): 5238, np.int64(4): 658, np.int64(1): 88, np.int64(2): 9, np.int64(5): 6})
Scaled train sample:
 [[-0.44275696 -0.02255926 -0.03631965 -0.01595383 -0.6334574  -0.25269275
  -0.61412135 -0.6889498  -1.12207725 -1.14167652 -0.395817   -0.2407031
  -0.23760612 -0.26712494 -0.34006479 -0.40762695 -0.25293704 -0.21369448
  -0.31398623 -0.22874386 -0.17194411 -0.17506571 -0.20451165 -0.08681352
  -0.07620395 -0.00521251 -0.17144919 -1.06044376 -0.43805359 -0.06941556
  -1.13102652 -0.0073717   1.76994901 -0.00368577 -0.297886    0.56361498
  -0.13980224 -0.10533779 -0.15013361 -0.1131472  -0.30811685 -0.12892585]
 [-0.44083347 -0.02193459 -0.03631965 -0.01595383 -0.55325852 -0.25269275
  -0.50416719 -0.54513146 -1.12207725 -1.14167652 -0.3953708  -0.

In [10]:
from imblearn.over_sampling import SMOTE
import collections

# Check minimum class size in training set
min_class_size = min(collections.Counter(y_train).values())
print("Smallest class size in train:", min_class_size)

# k_neighbors must be less than the smallest class size
k = min(5, min_class_size - 1)
print("Using k_neighbors =", k)

smote = SMOTE(random_state=42, k_neighbors=k)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Original train shape:", X_train_scaled.shape)
print("Resampled train shape:", X_train_resampled.shape)
print("Resampled class distribution:", collections.Counter(y_train_resampled))

Smallest class size in train: 22
Using k_neighbors = 5
Original train shape: (73612, 42)
Resampled train shape: (297690, 42)
Resampled class distribution: Counter({np.int64(0): 49615, np.int64(4): 49615, np.int64(3): 49615, np.int64(1): 49615, np.int64(2): 49615, np.int64(5): 49615})


In [11]:
import numpy as np

# Timing-related columns -> LSTM input (treat as a sequence of time-features)
timing_cols = [
    'Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
    'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
    'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
    'Active Mean', 'Active Std', 'Active Max', 'Active Min',
    'Idle Mean', 'Idle Std'
]

# keep only columns that actually exist after our earlier drop step
timing_cols = [c for c in timing_cols if c in X_reduced.columns]
static_cols = [c for c in X_reduced.columns if c not in timing_cols]

print("Timing cols (", len(timing_cols), "):", timing_cols)
print("Static cols (", len(static_cols), "):", static_cols)

Timing cols ( 19 ): ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std']
Static cols ( 23 ): ['Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Mean', 'Flow Byts/s', 'Flow Pkts/s', 'Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Pkts/s', 'Pkt Len Max', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'ACK Flag Cnt', 'CWE Flag Count', 'Down/Up Ratio', 'Init Bwd Win Byts']


In [12]:
import numpy as np

# Get column index positions from X_reduced (since X_train_scaled is a numpy array)
all_cols = list(X_reduced.columns)
timing_idx = [all_cols.index(c) for c in timing_cols]
static_idx = [all_cols.index(c) for c in static_cols]

# Use the SMOTE-resampled training data + scaled test data
X_train_timing = X_train_resampled[:, timing_idx]
X_train_static = X_train_resampled[:, static_idx]

X_test_timing = X_test_scaled[:, timing_idx]
X_test_static = X_test_scaled[:, static_idx]

# Reshape timing part for LSTM: (samples, timesteps=19, features=1)
X_train_lstm = X_train_timing.reshape(X_train_timing.shape[0], X_train_timing.shape[1], 1)
X_test_lstm = X_test_timing.reshape(X_test_timing.shape[0], X_test_timing.shape[1], 1)

print("LSTM train shape:", X_train_lstm.shape)
print("MLP train shape:", X_train_static.shape)
print("LSTM test shape:", X_test_lstm.shape)
print("MLP test shape:", X_test_static.shape)

# One-hot encode labels for multiclass output
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train_resampled)
y_test_cat = to_categorical(y_test)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)

LSTM train shape: (297690, 19, 1)
MLP train shape: (297690, 23)
LSTM test shape: (18404, 19, 1)
MLP test shape: (18404, 23)


2026-06-28 00:48:19.822576: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782607700.082075      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782607700.172972      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782607700.834358      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782607700.834411      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782607700.834414      58 computation_placer.cc:177] computation placer alr

y_train_cat shape: (297690, 6)
y_test_cat shape: (18404, 6)


In [13]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, concatenate, BatchNormalization
from tensorflow.keras.optimizers import Adam

# ---- LSTM branch (timing features) ----
lstm_input = Input(shape=(X_train_lstm.shape[1], 1), name='lstm_input')
x1 = LSTM(64, return_sequences=True)(lstm_input)
x1 = LSTM(32)(x1)
x1 = Dropout(0.3)(x1)

# ---- MLP branch (static features) ----
mlp_input = Input(shape=(X_train_static.shape[1],), name='mlp_input')
x2 = Dense(64, activation='relu')(mlp_input)
x2 = BatchNormalization()(x2)
x2 = Dropout(0.3)(x2)
x2 = Dense(32, activation='relu')(x2)

# ---- Merge both branches ----
merged = concatenate([x1, x2])
z = Dense(64, activation='relu')(merged)
z = Dropout(0.3)(z)
z = Dense(32, activation='relu')(z)
output = Dense(y_train_cat.shape[1], activation='softmax', name='output')(z)

model = Model(inputs=[lstm_input, mlp_input], outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

I0000 00:00:1782607718.316996      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782607718.323639      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mlp_input           │ (None, 23)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_input          │ (None, 19, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      1,536 │ mlp_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 19, 64)    │     16,896 │ lstm_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64)        │        256 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     12,416 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      4,160 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,080 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 6)         │        198 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 39,622 (154.77 KB)

 Trainable params: 39,494 (154.27 KB)

 Non-trainable params: 128 (512.00 B)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

history = model.fit(
    [X_train_lstm, X_train_static], y_train_cat,
    validation_split=0.1,
    epochs=30,
    batch_size=128,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

Epoch 1/30


I0000 00:00:1782607725.956963     130 cuda_dnn.cc:529] Loaded cuDNN version 91002


2094/2094 ━━━━━━━━━━━━━━━━━━━━ 34s 13ms/step - accuracy: 0.9061 - loss: 0.2798 - val_accuracy: 0.9316 - val_loss: 0.2231 - learning_rate: 0.0010
Epoch 2/30
2094/2094 ━━━━━━━━━━━━━━━━━━━━ 28s 13ms/step - accuracy: 0.9451 - loss: 0.1659 - val_accuracy: 0.9540 - val_loss: 0.1270 - learning_rate: 0.0010
Epoch 3/30
2094/2094 ━━━━━━━━━━━━━━━━━━━━ 28s 13ms/step - accuracy: 0.9524 - loss: 0.1438 - val_accuracy: 0.9808 - val_loss: 0.0895 - learning_rate: 0.0010
Epoch 4/30
2094/2094 ━━━━━━━━━━━━━━━━━━━━ 27s 13ms/step - accuracy: 0.9580 - loss: 0.1285 - val_accuracy: 0.9630 - val_loss: 0.1261 - learning_rate: 0.0010
Epoch 5/30
2094/2094 ━━━━━━━━━━━━━━━━━━━━ 27s 13ms/step - accuracy: 0.9623 - loss: 0.1154 - val_accuracy: 0.9620 - val_loss: 0.1099 - learning_rate: 0.0010
Epoch 6/30
2094/2094 ━━━━━━━━━━━━━━━━━━━━ 28s 13ms/step - accuracy: 0.9641 - loss: 0.1107 - val_accuracy: 0.9728 - val_loss: 0.0755 - learning_rate: 0.0010
Epoch 7/30
2094/2094 ━━━━━━━━━━━━━━━━━━━━ 28s 13ms/step - accuracy: 0.9655 

In [15]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Predict on test set
y_pred_probs = model.predict([X_test_lstm, X_test_static])
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_cat, axis=1)

# Classification report (precision, recall, f1 per class)
print(classification_report(y_true, y_pred, target_names=le.classes_, digits=4))

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

576/576 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
              precision    recall  f1-score   support

 Brute_Force     0.9975    0.9938    0.9956     12405
   HTTP_DDoS     0.9398    0.8864    0.9123        88
  ICMP_Flood     0.0617    0.5556    0.1111         9
      Normal     0.9953    0.9742    0.9847      5238
   Port_Scan     0.8817    0.9514    0.9152       658
 Web_Crwling     0.0455    0.3333    0.0800         6

    accuracy                         0.9858     18404
   macro avg     0.6536    0.7824    0.6665     18404
weighted avg     0.9917    0.9858    0.9885     18404

Confusion Matrix:
[[12328     3     0    13    61     0]
 [    1    78     5     0     4     0]
 [    0     0     5     3     0     1]
 [   12     0    67  5103    19    37]
 [   18     2     2     6   626     4]
 [    0     0     2     2     0     2]]


In [16]:
# Save the trained model
model.save('lstm_mlp_ids_model.h5')

# Save the scaler and label encoder for future inference
import joblib
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')

# Save classification report + confusion matrix as a text file for your report
with open('evaluation_report.txt', 'w') as f:
    f.write(classification_report(y_true, y_pred, target_names=le.classes_, digits=4))
    f.write("\n\nConfusion Matrix:\n")
    f.write(str(confusion_matrix(y_true, y_pred)))

print("All artifacts saved.")

All artifacts saved.


NameError: name 'xgb_model' is not defined